# Geração local de casos Gherkin em JSON — Mistral 7B Instruct v0.3 / RunPod — 3 técnicas

Este notebook foi adaptado para execução no **RunPod/Jupyter**.

Ele:

1. recebe uma planilha CSV ou XLSX disponível no ambiente do RunPod;
2. usa somente as colunas `id` e `name`;
3. baixa e executa o `mistralai/Mistral-7B-Instruct-v0.3` localmente na GPU do Pod;
4. executa, em sequência, as três técnicas: **zero-shot**, **one-shot** e **few-shot**;
5. executa cada caso a quantidade de vezes configurada em cada técnica;
6. gera **um JSON independente para cada técnica**;
7. salva o progresso após cada caso, permitindo retomar uma execução interrompida;
8. mantém o modelo carregado uma única vez durante as três técnicas;
9. monta o JSON no código, sem pedir ao modelo que produza JSON.

Arquivos produzidos:

- `geracoes_gherkin_<modelo>_zero-shot.json`
- `geracoes_gherkin_<modelo>_one-shot.json`
- `geracoes_gherkin_<modelo>_few-shot.json`
- `geracoes_gherkin_<modelo>_3_tecnicas.zip`

Formato exigido da saída do modelo:

```gherkin
Scenario: [Descrição do cenário]
  Given [contexto inicial]
    And [contexto adicional, quando necessário]
  When [ação realizada]
  Then [resultado esperado]
    And [resultado adicional, quando necessário]
```

> A inferência ocorre localmente na GPU do RunPod, sem API de inferência. O modelo é carregado uma única vez e reutilizado nas três técnicas.


## 1. Instalação das dependências

In [ ]:
%pip install -q -U "transformers>=4.42,<6" accelerate bitsandbytes safetensors sentencepiece huggingface_hub pandas openpyxl tqdm


## 2. Importações e verificação da GPU

In [ ]:
import os

# ============================================================
# CONFIGURAÇÃO DO HUGGING FACE ANTES DOS IMPORTS
# ============================================================

# Usa download HTTP tradicional. Isso evita problemas de reconstrução
# via Xet em alguns Pods e não afeta a inferência local.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"

# Cache persistente recomendado no RunPod.
os.environ["HF_HOME"] = "/workspace/huggingface"

# Ajuda a reduzir fragmentação de VRAM.
os.environ.setdefault(
    "PYTORCH_CUDA_ALLOC_CONF",
    "expandable_segments:True"
)

import re
import json
import time
import shutil
from pathlib import Path
from typing import Optional

import pandas as pd
import torch
import transformers
import bitsandbytes as bnb
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("bitsandbytes:", bnb.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    total_memory = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    capability = torch.cuda.get_device_capability(0)
    print(f"Memória total da GPU: {total_memory:.1f} GB")
    print("Compute Capability:", capability)
else:
    raise RuntimeError("Este notebook exige uma GPU CUDA no Pod do RunPod.")

workspace_usage = shutil.disk_usage("/workspace")
workspace_free_gb = workspace_usage.free / (1024 ** 3)
print(f"Espaço livre em /workspace: {workspace_free_gb:.1f} GB")

if workspace_free_gb < 18:
    raise RuntimeError(
        "Há menos de 18 GB livres em /workspace. "
        "Libere espaço antes de baixar o Mistral 7B."
    )


## 3. Localização da planilha no RunPod

In [ ]:
# Envie a planilha pelo navegador de arquivos do Jupyter.
# O diretório persistente recomendado no RunPod é /workspace.
#
# Exemplos:
#   /workspace/casos.csv
#   /workspace/casos.xlsx

INPUT_FILE = "/workspace/casos.csv"

if not Path(INPUT_FILE).exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {INPUT_FILE}\n"
        "Envie a planilha pelo Jupyter e ajuste INPUT_FILE para o caminho correto."
    )

print("Arquivo selecionado:", INPUT_FILE)


## 4. Configuração do experimento

In [ ]:
# ------------------------------------------------------------------
# MODELO LOCAL
# ------------------------------------------------------------------

# Mistral AI — modelo instruction-tuned de 7B parâmetros.
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

# Pode ser também um caminho local já baixado:
# MODEL_ID = "/workspace/modelos/meu_modelo"

# ------------------------------------------------------------------
# EXPERIMENTO
# ------------------------------------------------------------------

# As três técnicas serão executadas nesta ordem.
TECHNIQUES = [
    "zero-shot",
    "one-shot",
    "few-shot",
]

# Quantas vezes CADA caso será enviado ao modelo EM CADA TÉCNICA.
# Mantido igual aos demais notebooks para comparabilidade experimental.
NUM_EXECUTIONS = 10

# None processa todos os casos da planilha.
# Use 200 para processar apenas os primeiros 200 casos.
NUM_CASES = None

# Parâmetros da geração.
# Mantidos iguais aos demais notebooks do experimento.
TEMPERATURE = 0.5
TOP_P = 0.8
MAX_NEW_TOKENS = 256
MAX_RETRIES = 3
BASE_SEED = 42

# Quantização local.
# Mantida em NF4/4-bit seguindo o padrão do notebook Granite.
USE_4BIT = True
TRUST_REMOTE_CODE = False

# Diretório persistente dos resultados no RunPod.
OUTPUT_DIR = Path("/workspace/resultados_gherkin")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if NUM_EXECUTIONS < 1:
    raise ValueError("NUM_EXECUTIONS deve ser maior ou igual a 1.")

if NUM_CASES is not None and NUM_CASES < 1:
    raise ValueError("NUM_CASES deve ser None ou maior ou igual a 1.")

print("Modelo:", MODEL_ID)
print("Técnicas:", ", ".join(TECHNIQUES))
print("Execuções por caso/técnica:", NUM_EXECUTIONS)
print("Diretório de saída:", OUTPUT_DIR)


## 5. Prompt Zero-shot

In [ ]:
ZERO_SHOT_PROMPT = """
Converta a seguinte descrição de caso de teste em um único cenário BDD, usando a sintaxe estrita de Gherkin. Certifique-se de que a saída contenha apenas a sintaxe de Gherkin para o cenário, sem comentários, explicações ou a palavra "Feature". Use o português para as descrições dos casos de teste e detalhes do cenário, mas mantenha as palavras-chave do Gherkin em inglês.

Agora, converta a seguinte descrição de caso de teste em exatamente um cenário BDD usando a sintaxe estrita de Gherkin. A saída deve seguir exatamente o formato do exemplo fornecido e não conter nada além do cenário BDD. Somente as palavras-chave do Gherkin devem estar em inglês; todo o outro texto deve estar em português. Se o caso de teste fornecido estiver em inglês, traduza-o para o português na geração do cenário BDD, mantendo as palavras-chave (Given, When, Then, And) em inglês.

Descrição do Caso de Teste:
{test_case}

Para um bom cenário BDD, certifique-se de declarar claramente o valor de negócio ou resultado esperado e mantenha o foco em uma única ação e seu resultado. Use apenas os passos essenciais (Given, When, Then, And) de forma clara e declarativa, evitando detalhes de implementação e repetições desnecessárias. Garanta que os cenários sejam independentes, utilizem uma terminologia de negócios consistente sem jargões técnicos e que os passos sejam escritos em terceira pessoa para evitar múltiplas interpretações.

Certifique-se de que os cenários BDD tenham indentação consistente de dois espaços para cada passo sob 'Scenario', sem linhas em branco entre os passos, e uma linha em branco separando diferentes cenários.

Siga esta estrutura para a saída:

Scenario: [Descrição do Cenário]
  Given [algum contexto inicial]
    And [mais algum contexto, se houver]
  When [uma ação é realizada]
  Then [um conjunto específico de resultados deve ocorrer]
    And [outro resultado, se houver]

A resposta deve conter estritamente apenas a sintaxe válida de Gherkin e evitar qualquer informação ou comentário extra. A resposta deve conter apenas o texto BDD, sem formatação ou texto adicional (por exemplo, sem 'gherkin```'), e deve ser apenas um cenário BDD.
""".strip()

## 6. Prompt One-shot

In [ ]:
ONE_SHOT_PROMPT = """
Converta a seguinte descrição de caso de teste em um único cenário BDD, usando a sintaxe estrita de Gherkin. Certifique-se de que a saída contenha apenas a sintaxe de Gherkin para o cenário, sem comentários, explicações ou a palavra "Feature". Use o português para as descrições dos casos de teste e detalhes do cenário, mas mantenha as palavras-chave do Gherkin em inglês.

# Exemplo
# Descrição do caso de teste:
# Usuário tenta login com credenciais inválidas.

Scenario: Login com senha inválida
  Given o usuário está na página de login
    And o usuário insere um nome de usuário válido
  When o usuário insere uma senha inválida e clica no botão de login
  Then o sistema exibe uma mensagem de erro indicando que a senha está incorreta
    And o campo de senha é limpo

Agora, converta a seguinte descrição de caso de teste em exatamente um cenário BDD usando a sintaxe estrita de Gherkin. A saída deve seguir exatamente o formato do exemplo fornecido e não conter nada além do cenário BDD. Somente as palavras-chave do Gherkin devem estar em inglês; todo o outro texto deve estar em português. Se o caso de teste fornecido estiver em inglês, traduza-o para o português na geração do cenário BDD, mantendo as palavras-chave (Given, When, Then, And) em inglês.

Descrição do Caso de Teste:
{test_case}

Para um bom cenário BDD, certifique-se de declarar claramente o valor de negócio ou resultado esperado e mantenha o foco em uma única ação e seu resultado. Use apenas os passos essenciais (Given, When, Then, And) de forma clara e declarativa, evitando detalhes de implementação e repetições desnecessárias. Garanta que os cenários sejam independentes, utilizem uma terminologia de negócios consistente sem jargões técnicos e que os passos sejam escritos em terceira pessoa para evitar múltiplas interpretações.
Certifique-se de que os cenários BDD tenham indentação consistente de dois espaços para cada passo sob 'Scenario', sem linhas em branco entre os passos, e uma linha em branco separando diferentes cenários.

Siga esta estrutura para a saída:

Scenario: [Descrição do Cenário]
  Given [algum contexto inicial]
    And [mais algum contexto, se houver]
  When [uma ação é realizada]
  Then [um conjunto específico de resultados deve ocorrer]
    And [outro resultado, se houver]

A resposta deve conter estritamente apenas a sintaxe válida de Gherkin e evitar qualquer informação ou comentário extra. A resposta deve conter apenas o texto BDD, sem formatação ou texto adicional (por exemplo, sem 'gherkin```'), e deve ser apenas um cenário BDD.
""".strip()

## 7. Prompt Few-shot

In [ ]:
FEW_SHOT_PROMPT = """
Converta a seguinte descrição de caso de teste em um único cenário BDD, usando a sintaxe estrita de Gherkin. Certifique-se de que a saída contenha apenas a sintaxe de Gherkin para o cenário, sem comentários, explicações ou a palavra "Feature". Use o português para as descrições dos casos de teste e detalhes do cenário, mas mantenha as palavras-chave do Gherkin em inglês.

# Exemplo 1
# Descrição do caso de teste:
# Usuário tenta login com credenciais inválidas.

Scenario: Login com senha inválida
  Given o usuário está na página de login
    And o usuário insere um nome de usuário válido
  When o usuário insere uma senha inválida e clica no botão de login
  Then o sistema exibe uma mensagem de erro indicando que a senha está incorreta
    And o campo de senha é limpo

# Exemplo 2
# Descrição do caso de teste:
# Usuário tenta redefinir a senha esquecida.

Scenario: Redefinição de senha com e-mail válido
  Given o usuário está na página de redefinição de senha
    And o usuário insere um endereço de e-mail registrado
  When o usuário clica no botão de enviar
  Then o sistema exibe uma mensagem indicando que um link de redefinição de senha foi enviado para o e-mail do usuário
    And o usuário é redirecionado para a página de login

# Exemplo 3
# Descrição do caso de teste:
# Usuário adiciona um item ao carrinho de compras.

Scenario: Adicionar item ao carrinho de compras
  Given o usuário está na página de detalhes de um produto
    And o produto está disponível em estoque
  When o usuário clica no botão "Adicionar ao carrinho"
  Then o item é adicionado ao carrinho de compras
    And o sistema exibe uma mensagem de confirmação "Item adicionado ao carrinho com sucesso"
    And o ícone do carrinho de compras é atualizado para refletir o novo item

Agora, converta a seguinte descrição de caso de teste em exatamente um cenário BDD usando a sintaxe estrita de Gherkin. A saída deve seguir exatamente o formato do exemplo fornecido e não conter nada além do cenário BDD. Somente as palavras-chave do Gherkin devem estar em inglês; todo o outro texto deve estar em português. Se o caso de teste fornecido estiver em inglês, traduza-o para o português na geração do cenário BDD, mantendo as palavras-chave (Given, When, Then, And) em inglês.

Descrição do Caso de Teste:
{test_case}

Para um bom cenário BDD, certifique-se de declarar claramente o valor de negócio ou resultado esperado e mantenha o foco em uma única ação e seu resultado. Use apenas os passos essenciais (Given, When, Then, And) de forma clara e declarativa, evitando detalhes de implementação e repetições desnecessárias. Garanta que os cenários sejam independentes, utilizem uma terminologia de negócios consistente sem jargões técnicos e que os passos sejam escritos em terceira pessoa para evitar múltiplas interpretações.
Certifique-se de que os cenários BDD tenham indentação consistente de dois espaços para cada passo sob 'Scenario', sem linhas em branco entre os passos, e uma linha em branco separando diferentes cenários.

Siga esta estrutura para a saída:

Scenario: [Descrição do Cenário]
  Given [algum contexto inicial]
    And [mais algum contexto, se houver]
  When [uma ação é realizada]
  Then [um conjunto específico de resultados deve ocorrer]
    And [outro resultado, se houver]

A resposta deve conter estritamente apenas a sintaxe válida de Gherkin e evitar qualquer informação ou comentário extra. A resposta deve conter apenas o texto BDD, sem formatação ou texto adicional (por exemplo, sem 'gherkin```'), e deve ser apenas um cenário BDD.
""".strip()

## 8. Registro dos prompts


In [ ]:
TECHNIQUE_ALIASES = {
    "zero": "zero-shot",
    "zero-shot": "zero-shot",
    "zero_shot": "zero-shot",
    "one": "one-shot",
    "one-shot": "one-shot",
    "one_shot": "one-shot",
    "few": "few-shot",
    "few-shot": "few-shot",
    "few_shot": "few-shot",
}

PROMPTS = {
    "zero-shot": ZERO_SHOT_PROMPT,
    "one-shot": ONE_SHOT_PROMPT,
    "few-shot": FEW_SHOT_PROMPT,
}

normalized_techniques = []

for technique in TECHNIQUES:
    normalized_input = str(technique).strip().lower()

    if normalized_input not in TECHNIQUE_ALIASES:
        raise ValueError(
            f"Técnica inválida: {technique}. "
            "Use zero-shot, one-shot ou few-shot."
        )

    normalized_techniques.append(TECHNIQUE_ALIASES[normalized_input])

TECHNIQUES = normalized_techniques

if len(set(TECHNIQUES)) != len(TECHNIQUES):
    raise ValueError("TECHNIQUES contém técnicas duplicadas.")

print("Técnicas configuradas:", TECHNIQUES)


## 9. Token do Hugging Face e carregamento local do Mistral


In [ ]:
# O Mistral 7B Instruct v0.3 é público no Hugging Face.
# O token não é obrigatório, mas é recomendado para aumentar os limites
# e melhorar a estabilidade do download.
#
# Opção 1: defina HF_TOKEN nas variáveis de ambiente do Pod.
# Opção 2: cole o token no campo oculto abaixo quando solicitado.

from getpass import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    print("HF_TOKEN encontrado nas variáveis de ambiente.")
else:
    typed_token = getpass(
        "HF_TOKEN (token Read; pressione Enter para continuar sem token): "
    ).strip()

    HF_TOKEN = typed_token or None

    if HF_TOKEN:
        os.environ["HF_TOKEN"] = HF_TOKEN
        print("HF_TOKEN configurado para esta sessão.")
    else:
        print(
            "Nenhum token informado. O modelo é público e o download continuará, "
            "mas o Hugging Face poderá aplicar limites menores."
        )


In [ ]:
if USE_4BIT and not torch.cuda.is_available():
    raise RuntimeError("A quantização em 4 bits exige uma GPU CUDA.")

if torch.cuda.is_available():
    major, _ = torch.cuda.get_device_capability()
    COMPUTE_DTYPE = torch.bfloat16 if major >= 8 else torch.float16
else:
    COMPUTE_DTYPE = torch.float32

quantization_config = None

if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

print("=" * 72)
print("CARREGAMENTO DO TOKENIZER")
print("=" * 72)
print("Modelo:", MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=TRUST_REMOTE_CODE,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer carregado.")
print("Chat template disponível:", bool(getattr(tokenizer, "chat_template", None)))

model_kwargs = {
    "device_map": "auto",
    "token": HF_TOKEN,
    "trust_remote_code": TRUST_REMOTE_CODE,
    "low_cpu_mem_usage": True,
}

if quantization_config is not None:
    model_kwargs["quantization_config"] = quantization_config
    model_kwargs["torch_dtype"] = COMPUTE_DTYPE
else:
    model_kwargs["torch_dtype"] = COMPUTE_DTYPE

print()
print("=" * 72)
print("CARREGAMENTO DO MODELO")
print("=" * 72)
print("Baixando/carregando pesos:", MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    **model_kwargs,
)

model.eval()

print()
print("Modelo carregado localmente.")
print("Quantização 4-bit:", USE_4BIT)
print("dtype de computação:", COMPUTE_DTYPE)
print("Mapa de dispositivos:", getattr(model, "hf_device_map", "dispositivo único"))

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / (1024 ** 3)
    reserved = torch.cuda.memory_reserved(0) / (1024 ** 3)
    total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

    print(f"VRAM alocada: {allocated:.2f} GB")
    print(f"VRAM reservada: {reserved:.2f} GB")
    print(f"VRAM total: {total:.2f} GB")


## 10. Funções de leitura, geração e validação

In [ ]:
def read_spreadsheet(file_path: str) -> pd.DataFrame:
    path = Path(file_path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        last_error = None

        for encoding in ("utf-8", "utf-8-sig", "latin-1"):
            try:
                return pd.read_csv(path, encoding=encoding)
            except UnicodeDecodeError as error:
                last_error = error

        raise last_error

    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)

    raise ValueError("A entrada deve ser um arquivo CSV, XLSX ou XLS.")


def normalize_source_id(value) -> str:
    if pd.isna(value):
        raise ValueError("Foi encontrado um caso sem valor na coluna 'id'.")

    try:
        numeric = float(value)
        if numeric.is_integer():
            return str(int(numeric))
    except (TypeError, ValueError):
        pass

    return str(value).strip()


def slugify(value: str, max_length: int = 80) -> str:
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9._-]+", "-", value)
    value = value.strip("-")
    return value[:max_length] or "item"


def build_chat_prompt(user_prompt: str) -> str:
    system_prompt = (
        "Você converte títulos de casos de teste em cenários Gherkin. "
        "Sua resposta deve conter somente um Scenario em Gherkin, "
        "sem JSON, Feature, tags, Markdown, comentários ou explicações."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    return (
        f"System:\n{system_prompt}\n\n"
        f"User:\n{user_prompt}\n\n"
        "Assistant:\n"
    )


def clean_gherkin(raw_text: str) -> str:
    if raw_text is None:
        raise ValueError("O modelo retornou None.")

    text = str(raw_text).strip()

    # Remove cercas Markdown caso o modelo desobedeça ao prompt.
    text = re.sub(r"^```(?:gherkin|text)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)

    # Mantém somente o conteúdo a partir do primeiro Scenario.
    match = re.search(r"(?im)^\s*Scenario\s*:", text)

    if not match:
        raise ValueError("A resposta não contém 'Scenario:'.")

    text = text[match.start():]

    cleaned_lines = []
    scenario_count = 0

    for raw_line in text.splitlines():
        line = raw_line.strip()

        if not line:
            continue

        if re.match(r"(?i)^Feature\s*:", line):
            continue

        if line.startswith("@"):
            continue

        if re.match(r"(?i)^Scenario\s*:", line):
            scenario_count += 1
            if scenario_count > 1:
                break
            cleaned_lines.append(re.sub(r"(?i)^Scenario\s*:", "Scenario:", line))
            continue

        if re.match(r"(?i)^Given\b", line):
            cleaned_lines.append("  " + re.sub(r"(?i)^Given\b", "Given", line))
            continue

        if re.match(r"(?i)^When\b", line):
            cleaned_lines.append("  " + re.sub(r"(?i)^When\b", "When", line))
            continue

        if re.match(r"(?i)^Then\b", line):
            cleaned_lines.append("  " + re.sub(r"(?i)^Then\b", "Then", line))
            continue

        if re.match(r"(?i)^And\b", line):
            cleaned_lines.append("    " + re.sub(r"(?i)^And\b", "And", line))
            continue

        # Interrompe quando surgir texto explicativo após o cenário.
        if cleaned_lines:
            break

    result = "\n".join(cleaned_lines).strip()
    validate_gherkin(result)
    return result


def validate_gherkin(gherkin: str) -> None:
    required_patterns = {
        "Scenario": r"(?m)^Scenario\s*:",
        "Given": r"(?m)^\s{2}Given\b",
        "When": r"(?m)^\s{2}When\b",
        "Then": r"(?m)^\s{2}Then\b",
    }

    if len(re.findall(r"(?m)^Scenario\s*:", gherkin)) != 1:
        raise ValueError("A resposta deve conter exatamente um Scenario.")

    missing = [
        keyword
        for keyword, pattern in required_patterns.items()
        if not re.search(pattern, gherkin)
    ]

    if missing:
        raise ValueError(
            "A resposta não contém as palavras-chave obrigatórias: "
            + ", ".join(missing)
        )

    forbidden_patterns = [
        r"(?im)^Feature\s*:",
        r"(?m)^@",
        r"```",
        r"(?im)^\s*\{",
    ]

    if any(re.search(pattern, gherkin) for pattern in forbidden_patterns):
        raise ValueError("A resposta contém elementos proibidos.")


def generate_local_gherkin(
    test_case: str,
    execution_seed: int,
    prompt_template: str,
) -> str:
    user_prompt = prompt_template.format(test_case=test_case)
    formatted_prompt = build_chat_prompt(user_prompt)

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )

    input_device = model.get_input_embeddings().weight.device
    inputs = {key: value.to(input_device) for key, value in inputs.items()}

    input_length = inputs["input_ids"].shape[-1]
    set_seed(execution_seed)

    do_sample = TEMPERATURE > 0

    generation_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": do_sample,
        "top_p": TOP_P,
        "repetition_penalty": 1.05,
        "eos_token_id": tokenizer.eos_token_id,
        "pad_token_id": tokenizer.pad_token_id,
    }

    if do_sample:
        generation_kwargs["temperature"] = max(TEMPERATURE, 0.05)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            **generation_kwargs,
        )

    generated_tokens = outputs[0][input_length:]

    raw_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    return clean_gherkin(raw_text)


def save_json_atomic(data: dict, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = output_path.with_suffix(output_path.suffix + ".tmp")

    with temporary_path.open("w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )

    temporary_path.replace(output_path)

## 11. Preparação da planilha


In [ ]:
dataframe = read_spreadsheet(INPUT_FILE)

required_columns = {"id", "name"}
missing_columns = required_columns - set(dataframe.columns)

if missing_columns:
    raise ValueError(
        "A planilha não contém as colunas obrigatórias: "
        + ", ".join(sorted(missing_columns))
    )

if dataframe["id"].isna().any():
    raise ValueError("A coluna 'id' contém valores vazios.")

if dataframe["id"].duplicated().any():
    duplicated = dataframe.loc[
        dataframe["id"].duplicated(keep=False),
        "id",
    ].tolist()
    raise ValueError(f"A coluna 'id' contém valores duplicados: {duplicated[:10]}")

dataframe = dataframe.copy()
dataframe["source_id"] = dataframe["id"].apply(normalize_source_id)
dataframe["original_case"] = dataframe["name"].fillna("").astype(str).str.strip()

empty_names = dataframe["original_case"].eq("")

if empty_names.any():
    lines = (dataframe.index[empty_names] + 2).tolist()
    raise ValueError(
        f"Foram encontrados casos sem nome nas linhas: {lines[:10]}"
    )

# A linha 1 contém o cabeçalho; por isso o primeiro caso está na linha 2.
dataframe["source_line"] = dataframe.index + 2
dataframe["case_id"] = dataframe["source_id"].apply(
    lambda value: f"TC_{slugify(value)}"
)

if NUM_CASES is not None:
    if NUM_CASES > len(dataframe):
        raise ValueError(
            f"Foram solicitados {NUM_CASES} casos, "
            f"mas a planilha possui {len(dataframe)}."
        )
    selected_cases = dataframe.head(NUM_CASES).copy()
else:
    selected_cases = dataframe.copy()

model_slug = slugify(MODEL_ID.replace("/", "-"))

print("Casos na planilha:", len(dataframe))
print("Casos selecionados:", len(selected_cases))
print("Execuções por caso e por técnica:", NUM_EXECUTIONS)
print("Técnicas:", len(TECHNIQUES))
print(
    "Total máximo de gerações:",
    len(selected_cases) * NUM_EXECUTIONS * len(TECHNIQUES),
)


## 12. Execução sequencial: Zero-shot → One-shot → Few-shot


In [ ]:
def prepare_experiment(technique: str):
    output_path = OUTPUT_DIR / (
        f"geracoes_gherkin_{model_slug}_{technique}.json"
    )

    if output_path.exists():
        with output_path.open("r", encoding="utf-8") as file:
            experiment = json.load(file)

        compatibility_fields = {
            "model": MODEL_ID,
            "technique": technique,
            "source_file": Path(INPUT_FILE).name,
        }

        for field, expected_value in compatibility_fields.items():
            if experiment.get(field) != expected_value:
                raise ValueError(
                    f"O JSON existente {output_path.name} usa "
                    f"{field}={experiment.get(field)!r}, "
                    f"mas a configuração atual usa {expected_value!r}."
                )

        # Permite aumentar NUM_EXECUTIONS e continuar o mesmo arquivo.
        experiment["number_of_executions"] = NUM_EXECUTIONS
        print(f"[{technique}] JSON existente encontrado; retomando execução.")

    else:
        experiment = {
            "model": MODEL_ID,
            "technique": technique,
            "number_of_executions": NUM_EXECUTIONS,
            "source_file": Path(INPUT_FILE).name,
            "cases": [],
        }

    case_lookup = {
        case["case_id"]: case
        for case in experiment["cases"]
    }

    for _, row in selected_cases.iterrows():
        case_id = row["case_id"]

        if case_id not in case_lookup:
            case_record = {
                "case_id": case_id,
                "source_id": row["source_id"],
                "source_line": int(row["source_line"]),
                "original_case": row["original_case"],
                "generations": [],
            }
            experiment["cases"].append(case_record)
            case_lookup[case_id] = case_record
        else:
            # Mantém localização e texto original sincronizados.
            case_lookup[case_id]["source_id"] = row["source_id"]
            case_lookup[case_id]["source_line"] = int(row["source_line"])
            case_lookup[case_id]["original_case"] = row["original_case"]

    experiment["cases"].sort(key=lambda item: item["source_line"])
    save_json_atomic(experiment, output_path)

    return experiment, case_lookup, output_path


def run_technique(technique: str):
    prompt_template = PROMPTS[technique]
    experiment, case_lookup, output_path = prepare_experiment(technique)

    total_expected = len(selected_cases) * NUM_EXECUTIONS
    generated_now = 0
    skipped = 0
    failed = []

    progress = tqdm(
        total=total_expected,
        desc=f"{technique}",
    )

    for case_position, (_, row) in enumerate(selected_cases.iterrows()):
        case_id = row["case_id"]
        case_record = case_lookup[case_id]

        existing_generation_ids = {
            generation["generation_id"]
            for generation in case_record["generations"]
        }

        for execution in range(1, NUM_EXECUTIONS + 1):
            generation_id = (
                f"{case_id}__{model_slug}__"
                f"{technique}__exec-{execution:02d}"
            )

            if generation_id in existing_generation_ids:
                skipped += 1
                progress.update(1)
                continue

            last_error = None
            gherkin = None

            for attempt in range(1, MAX_RETRIES + 1):
                try:
                    # A mesma política de seeds é usada nas três técnicas.
                    seed = BASE_SEED + (case_position * 10_000) + execution

                    gherkin = generate_local_gherkin(
                        test_case=row["original_case"],
                        execution_seed=seed,
                        prompt_template=prompt_template,
                    )
                    break

                except Exception as error:
                    last_error = str(error)

                    if attempt < MAX_RETRIES:
                        time.sleep(2 ** (attempt - 1))

            if gherkin is None:
                failed.append({
                    "case_id": case_id,
                    "execution": execution,
                    "error": last_error,
                })
                progress.write(
                    f"Falha: {case_id}, execução {execution}: {last_error}"
                )
                progress.update(1)
                continue

            case_record["generations"].append({
                "generation_id": generation_id,
                "execution": execution,
                "gherkin": gherkin,
            })

            existing_generation_ids.add(generation_id)
            generated_now += 1
            progress.update(1)

        case_record["generations"].sort(
            key=lambda item: item["execution"]
        )

        experiment["cases"].sort(
            key=lambda item: item["source_line"]
        )

        # Salva após cada caso para permitir retomada.
        save_json_atomic(experiment, output_path)

    progress.close()

    print(f"\n[{technique}] concluído.")
    print("Novas gerações:", generated_now)
    print("Gerações já existentes ignoradas:", skipped)
    print("Falhas:", len(failed))
    print("JSON:", output_path)

    if failed:
        print("\nPrimeiras falhas:")
        for failure in failed[:20]:
            print(failure)

    return output_path


OUTPUT_PATHS = []

for technique in TECHNIQUES:
    print("\n" + "=" * 80)
    print("INICIANDO TÉCNICA:", technique)
    print("=" * 80)

    output_path = run_technique(technique)
    OUTPUT_PATHS.append(output_path)

print("\nTodas as técnicas foram processadas.")


## 13. Conferência dos três JSONs


In [ ]:
for output_path in OUTPUT_PATHS:
    with output_path.open("r", encoding="utf-8") as file:
        result = json.load(file)

    total_generations = sum(
        len(case["generations"])
        for case in result["cases"]
    )

    print("\nArquivo:", output_path.name)
    print("Modelo:", result["model"])
    print("Técnica:", result["technique"])
    print("Execuções configuradas:", result["number_of_executions"])
    print("Casos registrados:", len(result["cases"]))
    print("Gerações registradas:", total_generations)


## Estrutura dos arquivos gerados

Cada técnica gera um JSON independente no mesmo formato:

```json
{
  "model": "mistralai/Mistral-7B-Instruct-v0.3",
  "technique": "zero-shot",
  "number_of_executions": 10,
  "source_file": "casos.csv",
  "cases": [
    {
      "case_id": "TC_1",
      "source_id": "1",
      "source_line": 2,
      "original_case": "Cadastrar Entidade com sucesso",
      "generations": [
        {
          "generation_id": "TC_1__mistralai-mistral-7b-instruct-v0.3__zero-shot__exec-01",
          "execution": 1,
          "gherkin": "Scenario: ...\n  Given ...\n  When ...\n  Then ..."
        }
      ]
    }
  ]
}
```

A diferença entre os três arquivos está no campo `technique` e nas gerações produzidas pelo prompt correspondente.


## 14. Compactar os três JSONs em ZIP


In [ ]:
import zipfile

ZIP_PATH = OUTPUT_DIR / (
    f"geracoes_gherkin_{model_slug}_3_tecnicas.zip"
)

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zip_file:
    for path in OUTPUT_PATHS:
        zip_file.write(path, arcname=path.name)

print("JSONs disponíveis individualmente em:", OUTPUT_DIR)

for path in OUTPUT_PATHS:
    print(" -", path)

print("ZIP com os três resultados:", ZIP_PATH)
print("Os arquivos permanecem em /workspace e podem ser baixados pelo navegador de arquivos do Jupyter.")
